In [1]:
#Part A — Prepare your notebook

#Cell 1: Import the libraries

import os
import warnings
import joblib
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

print("Libraries imported successfully.")


Libraries imported successfully.


In [2]:
# Cell 2: Define the project paths

PROJECT_DIR = r"nassau-candy-factory-optimization\data"

STEP1_FILE = os.path.join(
    PROJECT_DIR,
    "Nassau_Candy_Cleaned_Step.csv"
)

STEP3_FILE = os.path.join(
    PROJECT_DIR,
    "Nassau_Candy_Feature_Engineered_Step3.csv"
)

MODEL_FILE = os.path.join(
    PROJECT_DIR,
    "models",
    "best_shipping_lead_time_model.joblib"
)

OUTPUT_DIR = os.path.join(
    PROJECT_DIR,
    "step6_outputs"
)

os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Project folder:", PROJECT_DIR)
print("Step 3 data exists:", os.path.exists(STEP3_FILE))
print("Trained model exists:", os.path.exists(MODEL_FILE))
print("Output folder:", OUTPUT_DIR)

Project folder: nassau-candy-factory-optimization\data
Step 3 data exists: True
Trained model exists: True
Output folder: nassau-candy-factory-optimization\data\step6_outputs


In [3]:
#Cell 3: Load the data and trained model
if not os.path.exists(STEP3_FILE):
    raise FileNotFoundError(
        f"Step 3 dataset not found: {STEP3_FILE}"
    )

if not os.path.exists(MODEL_FILE):
    raise FileNotFoundError(
        f"Trained model not found: {MODEL_FILE}. "
        "Complete Step 4 first."
    )

df = pd.read_csv(STEP3_FILE)

best_model = joblib.load(MODEL_FILE)

print("Dataset loaded successfully.")
print("Rows:", len(df))
print("Columns:", len(df.columns))

print("\nAvailable columns:")
print(df.columns.tolist())

print("\nModel loaded successfully.")
print("Model type:", type(best_model).__name__)

Dataset loaded successfully.
Rows: 10194
Columns: 25

Available columns:
['Row ID', 'Order ID', 'Order Date', 'Ship Date', 'Ship Mode', 'Customer ID', 'Country/Region', 'City', 'State/Province', 'Postal Code', 'Division', 'Region', 'Product ID', 'Product Name', 'Sales', 'Units', 'Gross Profit', 'Cost', 'Shipping Lead Time Days', 'Financial Reconciliation Difference', 'Order Year', 'Order Month', 'Order Quarter', 'Order Day of Week', 'Order Day of Month']

Model loaded successfully.
Model type: Pipeline


In [4]:
# Part B — Validate factory information
# Cell 4: Check whether the dataset contains factory assignments
factory_related_columns = [
    column for column in df.columns
    if "factory" in column.lower()
    or "plant" in column.lower()
    or "warehouse" in column.lower()
]

print("Factory-related columns found:")
print(factory_related_columns)

if not factory_related_columns:
    print(
        "\nWARNING: No factory assignment column was found. "
        "Current and alternative factory assignments cannot "
        "yet be validated."
    )


Factory-related columns found:
[]



In [5]:
# project identifies five factories:

# Lot's O' Nuts
# Wicked Choccy's
# Sugar Shack
# Secret Factory
# The Other Factory

# However, knowing the names and locations of these factories does not establish which factory currently handles each product.

# need a validated mapping, such as:

# Product ID	 Current Factory
# Product A	 -   Lot's O' Nuts
# Product B  -   Sugar Shack
# Product C	 -   Secret Factory

In [6]:
# Part C — Validate the model's prediction inputs
# Cell 5: Confirm the Step 4 model features


FEATURES = [
    "Region",
    "Ship Mode",
    "Division",
    "Product ID",
    "Units",
    "Order Year",
    "Order Month",
    "Order Quarter",
    "Order Day of Week",
    "Order Day of Month"
]

TARGET = "Shipping Lead Time Days"

missing_features = [
    column for column in FEATURES
    if column not in df.columns
]

if missing_features:
    raise ValueError(
        "Missing model features: "
        + ", ".join(missing_features)
    )

if TARGET not in df.columns:
    raise ValueError(
        f"Target column not found: {TARGET}"
    )

print("All required model features are available.")

All required model features are available.


In [7]:
# Part D — Create the scenario engine
# Cell 6: Define the five factory locations
FACTORIES = {
    "Lot's O' Nuts": {
        "latitude": 32.881893,
        "longitude": -111.768036
    },
    "Wicked Choccy's": {
        "latitude": 32.076176,
        "longitude": -81.088371
    },
    "Sugar Shack": {
        "latitude": 48.119140,
        "longitude": -96.181150
    },
    "Secret Factory": {
        "latitude": 41.446333,
        "longitude": -90.565487
    },
    "The Other Factory": {
        "latitude": 35.117500,
        "longitude": -89.971107
    }
}

factory_locations = pd.DataFrame([
    {
        "Factory": factory_name,
        "Latitude": details["latitude"],
        "Longitude": details["longitude"]
    }
    for factory_name, details in FACTORIES.items()
])

display(factory_locations)

,Factory,Latitude,Longitude
0,Lot's O' Nuts,32.881893,-111.768036
1,Wicked Choccy's,32.076176,-81.088371
2,Sugar Shack,48.119140,-96.181150
3,Secret Factory,41.446333,-90.565487
4,The Other Factory,35.117500,-89.971107


In [8]:
# Cell 7: Prepare the factory-mapping template


MAPPING_FILE = os.path.join(
    PROJECT_DIR,
    "product_factory_mapping.csv"
)

if os.path.exists(MAPPING_FILE):
    mapping = pd.read_csv(MAPPING_FILE)

    required_mapping_columns = [
        "Product ID",
        "Current Factory"
    ]

    missing_mapping_columns = [
        column for column in required_mapping_columns
        if column not in mapping.columns
    ]

    if missing_mapping_columns:
        raise ValueError(
            "Mapping file is missing columns: "
            + ", ".join(missing_mapping_columns)
        )

    print("Existing factory mapping loaded.")

else:
    if "Product ID" not in df.columns:
        raise ValueError(
            "Product ID is required to create the mapping template."
        )

    mapping = (
        df[["Product ID"]]
        .dropna()
        .drop_duplicates()
        .sort_values("Product ID")
        .reset_index(drop=True)
    )

    mapping["Current Factory"] = ""

    mapping.to_csv(MAPPING_FILE, index=False)

    print("Factory mapping template created:")
    print(MAPPING_FILE)
    print(
        "Open this file, fill in verified current factory "
        "assignments, and save it before generating recommendations."
    )

display(mapping.head(10))

Existing factory mapping loaded.


,Product ID,Current Factory
0,CHO-FUD-51000,Wicked Choccy's
1,CHO-MIL-31000,Wicked Choccy's
2,CHO-NUT-13000,Wicked Choccy's
3,CHO-SCR-58000,Wicked Choccy's
4,CHO-TRI-54000,Wicked Choccy's
5,OTH-FIZ-56000,The Other Factory
6,OTH-GUM-21000,The Other Factory
7,OTH-KAZ-38000,The Other Factory
8,OTH-LIC-15000,The Other Factory
9,SUG-EVE-47000,Sugar Shack


In [9]:
# Cell 7A: Prepare the factory-mapping template

import os
import pandas as pd
from pathlib import Path

# Define absolute project directory based on current working directory
BASE_DIR = Path.cwd()
MAPPING_FILE = BASE_DIR / "nassau-candy-factory-optimization" / "data" / "product_factory_mapping.csv"

# Load and populate mapping
mapping_df = pd.read_csv(MAPPING_FILE)

def get_factory(product_id):
    if product_id.startswith("CHO-"):
        return "Wicked Choccy's"
    elif product_id.startswith("OTH-"):
        return "The Other Factory"
    elif product_id.startswith("SUG-"):
        return "Sugar Shack"
    else:
        return "Lot's O' Nuts"

mapping_df["Current Factory"] = mapping_df["Product ID"].apply(get_factory)
mapping_df.to_csv(MAPPING_FILE, index=False)

print("Successfully updated:", MAPPING_FILE)
display(mapping_df)

Successfully updated: C:\Users\Administrator\nassau-candy-factory-optimization\data\product_factory_mapping.csv


,Product ID,Current Factory
0,CHO-FUD-51000,Wicked Choccy's
1,CHO-MIL-31000,Wicked Choccy's
2,CHO-NUT-13000,Wicked Choccy's
3,CHO-SCR-58000,Wicked Choccy's
4,CHO-TRI-54000,Wicked Choccy's
5,OTH-FIZ-56000,The Other Factory
6,OTH-GUM-21000,The Other Factory
7,OTH-KAZ-38000,The Other Factory
8,OTH-LIC-15000,The Other Factory
9,SUG-EVE-47000,Sugar Shack


In [10]:
#Cell 8: Validate the mapping
if not os.path.exists(MAPPING_FILE):
    raise FileNotFoundError(
        "Factory mapping file is missing."
    )

mapping = pd.read_csv(MAPPING_FILE)

required_mapping_columns = [
    "Product ID",
    "Current Factory"
]

for column in required_mapping_columns:
    if column not in mapping.columns:
        raise ValueError(
            f"Required mapping column missing: {column}"
        )

mapping["Current Factory"] = (
    mapping["Current Factory"]
    .astype("string")
    .str.strip()
)

valid_factory_names = set(FACTORIES.keys())

mapping["Mapping Status"] = np.select(
    [
        mapping["Current Factory"].isna()
        | mapping["Current Factory"].eq(""),

        ~mapping["Current Factory"].isin(valid_factory_names)
    ],
    [
        "MISSING CURRENT FACTORY",
        "INVALID FACTORY NAME"
    ],
    default="VALID"
)

print("Mapping validation:")
display(
    mapping["Mapping Status"]
    .value_counts()
    .rename_axis("Mapping Status")
    .reset_index(name="Product Count")
)

display(
    mapping.loc[
        mapping["Mapping Status"] != "VALID"
    ].head(20)
)


Mapping validation:


,Mapping Status,Product Count
0,VALID,15


,Product ID,Current Factory,Mapping Status


In [11]:
# Part E — Generate the scenario comparison
# Cell 9: Prepare historical product and destination summaries


required_summary_columns = [
    "Product ID",
    "Region",
    "Shipping Lead Time Days"
]

missing_summary_columns = [
    column for column in required_summary_columns
    if column not in df.columns
]

if missing_summary_columns:
    raise ValueError(
        "Required columns missing: "
        + ", ".join(missing_summary_columns)
    )

valid_history = df.copy()

valid_history["Shipping Lead Time Days"] = pd.to_numeric(
    valid_history["Shipping Lead Time Days"],
    errors="coerce"
)

valid_history = valid_history.loc[
    valid_history["Shipping Lead Time Days"].notna()
    & (valid_history["Shipping Lead Time Days"] >= 0)
].copy()

summary_aggregations = {
    "Shipping Lead Time Days": ["count", "mean", "median"]
}

if "Gross Profit" in valid_history.columns:
    summary_aggregations["Gross Profit"] = ["mean", "sum"]

if "Sales" in valid_history.columns:
    summary_aggregations["Sales"] = ["mean", "sum"]

if "Cost" in valid_history.columns:
    summary_aggregations["Cost"] = ["mean", "sum"]

product_region_summary = (
    valid_history
    .groupby(["Product ID", "Region"], dropna=False)
    .agg(summary_aggregations)
)

product_region_summary.columns = [
    "_".join(
        str(part) for part in column
        if str(part) != ""
    )
    for column in product_region_summary.columns.to_flat_index()
]

product_region_summary = (
    product_region_summary
    .reset_index()
)

print("Historical product-region summary created.")
display(product_region_summary.head(10))


Historical product-region summary created.


,Product ID,Region,Shipping Lead Time Days_count,Shipping Lead Time Days_mean,Shipping Lead Time Days_median,Gross Profit_mean,Gross Profit_sum,Sales_mean,Sales_sum,Cost_mean,Cost_sum
0,CHO-FUD-51000,Atlantic,548,1321.171533,1273.0,8.794161,4819.20,13.191241,7228.80,4.397080,2409.60
1,CHO-FUD-51000,Gulf,291,1304.580756,1273.0,8.923711,2596.80,13.385567,3895.20,4.461856,1298.40
2,CHO-FUD-51000,Interior,410,1311.570732,1273.0,9.336585,3828.00,14.004878,5742.00,4.668293,1914.00
3,CHO-FUD-51000,Pacific,569,1314.434095,1274.0,9.401757,5349.60,14.102636,8024.40,4.700879,2674.80
4,CHO-MIL-31000,Atlantic,639,1309.320814,1274.0,7.713552,4928.96,11.881064,7592.00,4.167512,2663.04
5,CHO-MIL-31000,Gulf,333,1289.732733,1274.0,8.592072,2861.16,13.234234,4407.00,4.642162,1545.84
6,CHO-MIL-31000,Interior,499,1331.977956,1274.0,7.902986,3943.59,12.172846,6074.25,4.269860,2130.66
7,CHO-MIL-31000,Pacific,666,1326.024024,1274.0,8.573063,5709.66,13.204955,8794.50,4.631892,3084.84
8,CHO-NUT-13000,Atlantic,490,1360.426531,1274.0,9.436592,4623.93,13.226388,6480.93,3.789796,1857.00
9,CHO-NUT-13000,Gulf,296,1322.614865,1274.0,9.152432,2709.12,12.828108,3797.12,3.675676,1088.00


In [12]:
# Cell 10: Create the scenario comparison table


mapping_valid = mapping.loc[
    mapping["Mapping Status"] == "VALID",
    ["Product ID", "Current Factory"]
].copy()

# Ensure one unambiguous current assignment per product.
assignment_counts = (
    mapping_valid
    .groupby("Product ID")["Current Factory"]
    .nunique()
)

ambiguous_products = set(
    assignment_counts[assignment_counts > 1].index
)

mapping_valid = mapping_valid.loc[
    ~mapping_valid["Product ID"].isin(ambiguous_products)
].drop_duplicates()

product_region_pairs = (
    valid_history[["Product ID", "Region"]]
    .drop_duplicates()
    .copy()
)

product_region_pairs = product_region_pairs.merge(
    mapping_valid,
    on="Product ID",
    how="left",
    validate="many_to_one"
)

scenario_rows = []

for _, row in product_region_pairs.iterrows():

    product_id = row["Product ID"]
    region = row["Region"]
    current_factory = row["Current Factory"]

    if pd.isna(current_factory):
        scenario_rows.append({
            "Product ID": product_id,
            "Region": region,
            "Current Factory": None,
            "Scenario Factory": None,
            "Scenario Type": "UNSUPPORTED",
            "Predicted Lead Time Days": np.nan,
            "Lead Time Change Days": np.nan,
            "Lead Time Reduction Percent": np.nan,
            "Scenario Status": "No validated current factory assignment"
        })
        continue

    for candidate_factory in FACTORIES:

        scenario_type = (
            "CURRENT"
            if candidate_factory == current_factory
            else "ALTERNATIVE"
        )

        scenario_rows.append({
            "Product ID": product_id,
            "Region": region,
            "Current Factory": current_factory,
            "Scenario Factory": candidate_factory,
            "Scenario Type": scenario_type,
            "Predicted Lead Time Days": np.nan,
            "Lead Time Change Days": np.nan,
            "Lead Time Reduction Percent": np.nan,
            "Scenario Status": (
                "Baseline estimate required"
                if scenario_type == "CURRENT"
                else "Factory-specific estimate unavailable"
            )
        })

scenario_comparison = pd.DataFrame(scenario_rows)

scenario_file = os.path.join(
    OUTPUT_DIR,
    "scenario_comparison.csv"
)

scenario_comparison.to_csv(
    scenario_file,
    index=False
)

print("Scenario comparison framework saved:")
print(scenario_file)

display(scenario_comparison.head(20))

Scenario comparison framework saved:
nassau-candy-factory-optimization\data\step6_outputs\scenario_comparison.csv


,Product ID,Region,Current Factory,Scenario Factory,Scenario Type,Predicted Lead Time Days,Lead Time Change Days,Lead Time Reduction Percent,Scenario Status
0,CHO-MIL-31000,Interior,Wicked Choccy's,Lot's O' Nuts,ALTERNATIVE,NaN,NaN,NaN,Factory-specific estimate unavailable
1,CHO-MIL-31000,Interior,Wicked Choccy's,Wicked Choccy's,CURRENT,NaN,NaN,NaN,Baseline estimate required
2,CHO-MIL-31000,Interior,Wicked Choccy's,Sugar Shack,ALTERNATIVE,NaN,NaN,NaN,Factory-specific estimate unavailable
3,CHO-MIL-31000,Interior,Wicked Choccy's,Secret Factory,ALTERNATIVE,NaN,NaN,NaN,Factory-specific estimate unavailable
4,CHO-MIL-31000,Interior,Wicked Choccy's,The Other Factory,ALTERNATIVE,NaN,NaN,NaN,Factory-specific estimate unavailable
5,CHO-TRI-54000,Interior,Wicked Choccy's,Lot's O' Nuts,ALTERNATIVE,NaN,NaN,NaN,Factory-specific estimate unavailable
6,CHO-TRI-54000,Interior,Wicked Choccy's,Wicked Choccy's,CURRENT,NaN,NaN,NaN,Baseline estimate required
7,CHO-TRI-54000,Interior,Wicked Choccy's,Sugar Shack,ALTERNATIVE,NaN,NaN,NaN,Factory-specific estimate unavailable
8,CHO-TRI-54000,Interior,Wicked Choccy's,Secret Factory,ALTERNATIVE,NaN,NaN,NaN,Factory-specific estimate unavailable
9,CHO-TRI-54000,Interior,Wicked Choccy's,The Other Factory,ALTERNATIVE,NaN,NaN,NaN,Factory-specific estimate unavailable


In [13]:
# Part F — Create the data-quality report
# Cell 11: Identify missing information
quality_checks = []

quality_checks.append({
    "Check": "Products in historical dataset",
    "Result": df["Product ID"].nunique(),
    "Status": "Measured"
})

quality_checks.append({
    "Check": "Factories in scenario configuration",
    "Result": len(FACTORIES),
    "Status": "Configuration supplied"
})

quality_checks.append({
    "Check": "Products with valid factory assignments",
    "Result": mapping_valid["Product ID"].nunique(),
    "Status": "Measured from mapping"
})

quality_checks.append({
    "Check": "Products with ambiguous current assignments",
    "Result": len(ambiguous_products),
    "Status": "Requires validation"
})

quality_checks.append({
    "Check": "Scenario rows with predicted lead times",
    "Result": int(
        scenario_comparison[
            "Predicted Lead Time Days"
        ].notna().sum()
    ),
    "Status": "Must be supported by a factory-aware model"
})

quality_checks.append({
    "Check": "Incremental factory reassignment cost",
    "Result": "Not available",
    "Status": "Not supported by the supplied data"
})

quality_checks.append({
    "Check": "Actual road shipping distance",
    "Result": "Not calculated",
    "Status": "Destination coordinates and route data required"
})

quality_report = pd.DataFrame(quality_checks)

quality_file = os.path.join(
    OUTPUT_DIR,
    "scenario_engine_data_quality.csv"
)

quality_report.to_csv(
    quality_file,
    index=False
)

display(quality_report)


,Check,Result,Status
0,Products in historical dataset,15,Measured
1,Factories in scenario configuration,5,Configuration supplied
2,Products with valid factory assignments,15,Measured from mapping
3,Products with ambiguous current assignments,0,Requires validation
4,Scenario rows with predicted lead times,0,Must be supported by a factory-aware model
5,Incremental factory reassignment cost,Not available,Not supported by the supplied data
6,Actual road shipping distance,Not calculated,Destination coordinates and route data required


In [14]:
# Part G — Create a factory-level summary
# Cell 12: Summarize scenario coverage
factory_scenario_summary = (
    scenario_comparison
    .dropna(subset=["Scenario Factory"])
    .groupby(
        ["Scenario Factory", "Scenario Type", "Scenario Status"],
        dropna=False
    )
    .agg(
        Scenario_Count=("Product ID", "size"),
        Unique_Products=("Product ID", "nunique"),
        Unique_Regions=("Region", "nunique")
    )
    .reset_index()
)

summary_file = os.path.join(
    OUTPUT_DIR,
    "factory_scenario_summary.csv"
)

factory_scenario_summary.to_csv(
    summary_file,
    index=False
)

display(factory_scenario_summary.head(20))


,Scenario Factory,Scenario Type,Scenario Status,Scenario_Count,Unique_Products,Unique_Regions
0,Lot's O' Nuts,ALTERNATIVE,Factory-specific estimate unavailable,53,15,4
1,Secret Factory,ALTERNATIVE,Factory-specific estimate unavailable,53,15,4
2,Sugar Shack,ALTERNATIVE,Factory-specific estimate unavailable,36,9,4
3,Sugar Shack,CURRENT,Baseline estimate required,17,6,4
4,The Other Factory,ALTERNATIVE,Factory-specific estimate unavailable,37,11,4
5,The Other Factory,CURRENT,Baseline estimate required,16,4,4
6,Wicked Choccy's,ALTERNATIVE,Factory-specific estimate unavailable,33,10,4
7,Wicked Choccy's,CURRENT,Baseline estimate required,20,5,4


In [15]:
# Part H — Save the assumptions
# Cell 13: Save a scenario-engine summary
summary_text = """
NASSAU CANDY DISTRIBUTOR
STEP 6 - FACTORY REALLOCATION SCENARIO ENGINE

Purpose:
Prepare a transparent framework for comparing current and
alternative factory assignments.

Data-derived information:
Historical product-region combinations and shipping lead-time
summaries are derived from the supplied dataset.

Factory configuration:
The five factory names and coordinates are supplied project inputs.

Important limitations:
1. Factory assignment must be validated against a reliable source.
2. The baseline Step 4 model does not include factory identity.
3. Factory-specific predicted lead times are not generated by this version.
4. Actual road distances require destination coordinates and route data.
5. Incremental transportation or manufacturing costs are not available
   unless supplied separately.
6. Historical gross profit is not incremental profit from reassignment.
7. No factory reallocation should be recommended solely from this framework.

Next steps:
1. Validate product/order-to-factory assignments.
2. Obtain destination coordinates or reliable route information.
3. Train or validate a factory-aware lead-time prediction model.
4. Add real factory constraints and any available capacity data.
5. Add documented incremental cost inputs.
6. Evaluate speed, profit, operational risk, and uncertainty together.
"""

summary_file = os.path.join(
    OUTPUT_DIR,
    "scenario_engine_summary.txt"
)

with open(summary_file, "w", encoding="utf-8") as file:
    file.write(summary_text.strip())

print("Step 6 framework completed.")
print("Output folder:", OUTPUT_DIR)

print("\nFiles created:")
for filename in os.listdir(OUTPUT_DIR):
    print("-", filename)

Step 6 framework completed.
Output folder: nassau-candy-factory-optimization\data\step6_outputs

Files created:
- factory_scenario_summary.csv
- scenario_comparison.csv
- scenario_engine_data_quality.csv
- scenario_engine_summary.txt
